In [111]:
!pip install -U pageindex openai python-dotenv

In [112]:
!pip install langchain-groq

In [113]:
import os
from dotenv import load_dotenv

load_dotenv()

env_content = """
PAGEINDEX_API_KEY=6564d555273742f9a0ec5b7bbc7558f3
GROQ_API_KEY=os.getenv('GROQ')
 """
with open(".env", "w") as f:
    f.write(env_content.strip())
print(".env file created")

.env file created


In [114]:
import os, json, time
from dotenv import load_dotenv
from google.colab import userdata

load_dotenv()

PAGEINDEX_API_KEY="496af133537c42e4a668c35e87b8e1e8"
GROQ_API_KEY=userdata.get("GROQ_API_KEY")

print("PageIndex key loaded:")
print("Groq key loaded:")

PageIndex key loaded:
Groq key loaded:


In [115]:
print("PAGEINDEX_API_KEY:", PAGEINDEX_API_KEY)
print("Length:", len(PAGEINDEX_API_KEY))

PAGEINDEX_API_KEY: 496af133537c42e4a668c35e87b8e1e8
Length: 32


In [116]:
from pageindex import PageIndexClient
from groq import Groq

pi_client=PageIndexClient(api_key=PAGEINDEX_API_KEY)
gq_client=Groq(api_key=GROQ_API_KEY)

print("PageIndex client ready")
print("Groq client ready")

PageIndex client ready
Groq client ready


In [117]:
PDF_PATH = "/content/sample.pdf"

print(f"Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"Uploaded!")
print(f"Document ID: {doc_id}")
print(" (Save this ID — you'll use it throughout the notebook)")

Uploading: /content/sample.pdf
Uploaded!
Document ID: pi-cmnx15wlm04so01pe0dw1rtzt
 (Save this ID — you'll use it throughout the notebook)


In [118]:
print("Building tree index...")
print(" (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f" Status: {status}")

    if status == "completed":
        print("\n Tree index ready!")
        break
    elif status == "failed":
        print("\nProcessing failed. Check your PDF format.")
        break

    time.sleep(5)

Building tree index...
 (This runs once per document — the index is cached for reuse)
 Status: processing
 Status: processing
 Status: completed

 Tree index ready!


In [119]:
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"Top-level sections: {len(pageindex_tree)}")
print("\n Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

Top-level sections: 1

 Raw tree (first node):
{
  "title": "Samadhan Kamble",
  "node_id": "0000",
  "page_index": 1,
  "summary": "This document outlines the profile of Samadhan Kamble, including his contact information, educational background with degrees and academic achievements, a comprehensive list of technical and soft skills, detailed descriptions of three key projects (Resume Analysis RAG System, AI Chatbot System, and Order Management System), relevant certifications in Artificial Intelligence & Machine Learning, Python, and Core Java, and proficiency in English, Hindi, and Marathi languages.",
  "text": "# Samadhan Kamble\n\nEmail: samadhank810@gmail.com\n\nMob: +917448101276\n\n|  Education : | Latur, Maharashtra, India  |\n| --- | --- |\n|  Dnyanshree Institute of Engineering & Technology | Satara  |\n|  B.Tech in Computer Science & Engineering | 2025  |\n|  Shivaji madhyamik Shala (SSC) | Latur  |\n|  Percentage : 86.80 | 2019  |\n|  Maharashtra Mahavidyalay (HSC) | Latu

In [120]:

def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + (" " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("Full Document Structure:\n")
print_tree(pageindex_tree)

Full Document Structure:

[0000] Samadhan Kamble  (p.1)


In [121]:
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

Total nodes in tree: 1
   Each node = one retrievable section of the document


In [122]:
import json

def llm_tree_search(query: str, tree: list, model: str = "llama-3.3-70b-versatile") -> dict:

    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title": n["title"],
                "page": n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    compressed_tree = compress(tree)

    prompt = f"""You are given a query and a document's tree structure.
Your task: identify which node IDs most likely contain the answer.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in valid JSON:
{{
  "thinking": "...",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = gq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )

    content = response.choices[0].message.content


    return json.loads(content)

In [123]:
import requests

try:
    r = requests.get("https://api.groq.com")
    print("Status:", r.status_code)
except Exception as e:
    print("Error:", e)

Status: 200


In [129]:
query = "give me  the phone number mentioned in pdf?"

print(f"Query: {query}\n")

result = llm_tree_search(query, pageindex_tree)

print("LLM Reasoning:")
print(result.get("thinking", "N/A"))

print("\nSelected Node IDs:")
print(result.get("node_list", []))

Query: give me  the phone number mentioned in pdf?

LLM Reasoning:
The query asks for the phone number mentioned in the PDF. The document tree contains a node with a phone number (+917448101276) in its summary. This suggests that the answer is likely to be found in this node.

Selected Node IDs:
['0000']


Key: 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
Type: <class 'str'>
